In [13]:
# %% [markdown]
# # 3.01 — Baseline Dummy

# %%
import sys
from pathlib import Path

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate

sys.path.insert(0, str(Path("..").resolve()))
from churn_telecom.config import (
    DATA_PROCESSED,
    RANDOM_STATE,
    REPORTS_FIGURES,
    get_logger,
    mlflow_run,
    setup_mlflow,
    log_dataset_to_mlflow,
)
from churn_telecom.plots import (
    save_confusion_matrix,
    save_roc_curve,
)

In [14]:
logger = get_logger("3.01_vab_baseline_dummy")
logger.info("Iniciando script 3.02_vab_baseline_logistic.py")

22:34:15 | INFO | Iniciando script 3.02_vab_baseline_logistic.py


In [15]:
TRAIN_PATH = DATA_PROCESSED / "train.parquet"
TEST_PATH = DATA_PROCESSED / "test.parquet"

assert TRAIN_PATH.exists(), f"Arquivo não encontrado: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Arquivo não encontrado: {TEST_PATH}"

train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)

TARGET = "churn_value"
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]
X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

logger.info("Train : %s | churn: %.2f%%", X_train.shape, y_train.mean() * 100)
logger.info("Test  : %s | churn: %.2f%%", X_test.shape, y_test.mean() * 100)

22:34:15 | INFO | Train : (5395, 30) | churn: 57.41%
22:34:15 | INFO | Test  : (1405, 30) | churn: 26.48%


In [16]:
X_train.head(5)

,tenure_months,monthly_charges,total_charges,num_services,charges_per_month,partner,dependents,phone_service,multiple_lines,online_security,...,contract_Two year,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,gender_Male,tenure_group_medio,tenure_group_novo,is_month_to_month,has_security_support,is_fiber_optic
0,0.219348,-1.702359,-0.468695,-1.161232,-0.974364,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
1,0.116770,-1.759988,-0.586644,-1.161232,-0.933337,1.0,1.0,1.0,0.0,0.0,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
2,0.311107,0.603372,0.487402,1.093836,-0.139046,1.0,1.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
3,0.738393,-1.388605,0.090235,-1.161232,-1.148963,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,1.085571,0.796309,1.175244,1.684347,-0.702402,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [18]:
setup_mlflow()

MODEL_NAME = "baseline-dummy"  # nome no Model Registry — sem barras nem pontos

with mlflow_run("3.01_baseline_dummy") as run:
    # 1. Tags descritivas
    mlflow.set_tags(
        {
            "notebook": "3.01",
            "fase": "baseline",
            "modelo": "dummy",
            "task": "classification",
            "framework": "sklearn",
        }
    )

    # 2. Dataset versionado (train + test)
    log_dataset_to_mlflow(X_train, y_train, split="train", source_path=TRAIN_PATH)
    log_dataset_to_mlflow(X_test, y_test, split="test", source_path=TEST_PATH)

    # 3. Parâmetros
    params = {
        "strategy": "most_frequent",
        "random_state": RANDOM_STATE,
        "cv_folds": 5,
    }
    mlflow.log_params(params)
    logger.info("Params: %s", params)

    # 4. Validação cruzada estratificada
    model = DummyClassifier(
        strategy=params["strategy"],
        random_state=params["random_state"],
    )
    cv = StratifiedKFold(
        n_splits=params["cv_folds"],
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=["average_precision","roc_auc", "f1", "recall", "precision"],
    )

    # 5. Treino final + predições no test set
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # 6. Métricas — técnicas + cross-val com desvio padrão
    metrics = {
        # Test set
        "test_auc": roc_auc_score(y_test, y_proba),
        "test_pr_auc": average_precision_score(y_test, y_proba),
        "test_f1": f1_score(y_test, y_pred, zero_division=0),
        "test_recall": recall_score(y_test, y_pred, zero_division=0),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        # Cross-val
        "cv_auc_mean": float(cv_results["test_roc_auc"].mean()),
        "cv_auc_std": float(cv_results["test_roc_auc"].std()),
        "cv_pr_auc_mean": float(cv_results["test_average_precision"].mean()),
        "cv_pr_auc_std": float(cv_results["test_average_precision"].std()),
        "cv_f1_mean": float(cv_results["test_f1"].mean()),
        "cv_f1_std": float(cv_results["test_f1"].std()),
        "cv_recall_mean": float(cv_results["test_recall"].mean()),
        "cv_recall_std": float(cv_results["test_recall"].std()),
    }
    mlflow.log_metrics(metrics)
    logger.info("Metrics: %s", {k: f"{v:.4f}" for k, v in metrics.items()})

    # 7. Artefatos visuais
    base = REPORTS_FIGURES / "baselines"

    mlflow.log_artifact(
        str(
            save_confusion_matrix(
                y_test,
                y_pred,
                base / "dummy_confusion_matrix.png",
                title="Confusion Matrix — Dummy",
            )
        ),
        artifact_path="baselines/dummy",
    )
    mlflow.log_artifact(
        str(
            save_roc_curve(
                y_test,
                y_proba,
                base / "dummy_roc_curve.png",
                model_name="Dummy",
                title="ROC Curve — Dummy",
            )
        ),
        artifact_path="baselines/dummy",
    )

    # 8. Log + registro no Model Registry
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",  # sem barras — compatível MLflow >= 2.18
        registered_model_name=MODEL_NAME,  # cria Version 1 no Registry
        input_example=X_test.iloc[:5],
    )

    logger.info("Model URI : %s", model_info.model_uri)
    logger.info("Run ID    : %s", run.info.run_id)

c:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\projeto_final\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\projeto_final\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns

Registered model 'baseline-dummy' already exists. Creating a new version of this model...
Created version '4' of model 'baseline-dummy'.
22:35:37 | INFO | Model URI : models:/m-abfead2a989f4a6baecc6c22b5d2b8c8
22:35:37 | INFO | Run ID    : e852593814eb4aaf91afd3845994ad8b
